# 04 · Analysis & Insights

**Goal:** answer real business questions with SQL + charts on the gold layer, and
state the decision each insight supports. (Maps to brief **§4.1 analytics**,
**§4.2 queries**, **§4.3 decision-making**.)

Each section below is a **screenshot target** for your report: a chart + a one-line
"so what". `display()` renders Databricks' built-in charts; we also draw matplotlib
figures you can paste into Word.

In [ ]:
%run ./00_config_and_setup

In [ ]:
import matplotlib.pyplot as plt
from pyspark.sql import functions as F

# Register gold tables as short SQL view names for tidy %sql cells
for t in ["daily_consumption", "daily_totals", "acorn_profile", "load_profile"]:
    spark.table(table("gold", t)).createOrReplaceTempView(t)

## Insight 1 — National demand trend over time
**Q:** How does total daily consumption move across the period?

In [ ]:
%sql
SELECT day, total_kwh, avg_kwh_per_home, active_homes
FROM daily_totals
ORDER BY day

In [ ]:
pdf = (spark.table(table("gold", "daily_totals"))
       .select("day", "avg_kwh_per_home").orderBy("day").toPandas())
plt.figure(figsize=(11, 4))
plt.plot(pdf["day"], pdf["avg_kwh_per_home"], lw=0.8)
plt.title("Average daily electricity use per household over time")
plt.ylabel("kWh / home / day"); plt.xlabel("Date"); plt.tight_layout()
plt.show()
# SO WHAT: clear winter peaks / summer troughs -> demand is seasonal & weather-driven,
# which justifies the forecasting model in notebook 05.

## Insight 2 — Who consumes most? (ACORN affluence group)
**Q:** Does affluence relate to electricity demand?

In [ ]:
%sql
SELECT acorn_group, ROUND(AVG(avg_daily_kwh),3) AS avg_daily_kwh, SUM(households) AS homes
FROM acorn_profile
WHERE acorn_group <> 'Unclassified'
GROUP BY acorn_group
ORDER BY avg_daily_kwh DESC

In [ ]:
# NOTE: this 21-block subset is almost entirely the 'Affluent' supergroup, so the 3-level
# acorn_group view collapses to one bar. We drill into the DETAILED ACORN category
# (ACORN-A..E), which varies within the subset and recovers a real affluence gradient.
pdf = (spark.table(table("gold", "daily_consumption"))
       .groupBy("acorn_code")
       .agg(F.avg("daily_kwh").alias("kwh"),
            F.countDistinct("LCLid").alias("homes"))
       .filter("homes >= 5")            # drop tiny groups for a clean chart
       .orderBy("kwh").toPandas())
plt.figure(figsize=(8, 4))
plt.barh(pdf["acorn_code"], pdf["kwh"], color="#2b8cbe")
plt.title("Average daily consumption by detailed ACORN category")
plt.xlabel("kWh / day"); plt.tight_layout(); plt.show()
print(pdf)
# SO WHAT: ACORN-A draws ~80% more than ACORN-E -> target efficiency incentives at the
# highest-consuming segments & inform differentiated tariff design.

## Insight 3 — Standard vs Time-of-Use tariff
**Q:** Do dynamic-priced (ToU) customers behave differently?

In [ ]:
%sql
SELECT tariff,
       ROUND(AVG(avg_daily_kwh),3) AS avg_daily_kwh,
       SUM(households)             AS households
FROM acorn_profile
GROUP BY tariff
ORDER BY avg_daily_kwh DESC

## Insight 4 — Weather sensitivity (temperature vs demand)
**Q:** How strongly does temperature drive consumption?

In [ ]:
corr = spark.table(table("gold", "daily_totals")).stat.corr("temp_avg", "avg_kwh_per_home")
print(f"Pearson correlation (avg temperature vs per-home kWh): {corr:.3f}")

pdf = (spark.table(table("gold", "daily_totals"))
       .select("temp_avg", "avg_kwh_per_home").dropna().toPandas())
plt.figure(figsize=(7, 5))
plt.scatter(pdf["temp_avg"], pdf["avg_kwh_per_home"], s=6, alpha=0.4)
plt.title(f"Demand vs temperature (r = {corr:.2f})")
plt.xlabel("Avg daily temperature (°C)"); plt.ylabel("kWh / home / day")
plt.tight_layout(); plt.show()
# SO WHAT: strong negative correlation = heating-led demand -> weather is a key
# feature for the forecast and for grid capacity planning.

## Insight 5 — Seasonal, weekend & holiday effects

In [ ]:
%sql
SELECT season,
       ROUND(AVG(avg_kwh_per_home),3)                                  AS avg_per_home,
       ROUND(AVG(CASE WHEN is_weekend=1 THEN avg_kwh_per_home END),3)  AS weekend,
       ROUND(AVG(CASE WHEN is_weekend=0 THEN avg_kwh_per_home END),3)  AS weekday,
       ROUND(AVG(CASE WHEN is_holiday=1 THEN avg_kwh_per_home END),3)  AS holiday
FROM daily_totals
GROUP BY season
ORDER BY avg_per_home DESC

## Insight 6 — The daily load curve (48 half-hours)
**Q:** When during the day does demand peak?

In [ ]:
pdf = (spark.table(table("gold", "load_profile"))
       .orderBy("half_hour").toPandas())
plt.figure(figsize=(11, 4))
plt.plot(pdf["clock_time"], pdf["avg_kwh"], marker="o", ms=3)
plt.title("Average half-hourly load profile (a typical day)")
plt.ylabel("kWh per half-hour"); plt.xlabel("Time of day")
plt.xticks(range(0, 48, 4), rotation=45); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()
# SO WHAT: the evening peak (~17:00-20:00) is where demand-response / dynamic pricing
# delivers most value -> directly informs ToU tariff windows.